# 1. Data
The entire text of plays: 1) The Tragedy of Hamlet, Prince of Denmark, 2) The Tragedy of Macbeth, and 3) The Tragedy of Julius Caesar. These are available from the Gutenberg corpus of the NLTK library. Characters and synopses can be found on Wikipedia.

**Problem Statement:**
Natural language processing is an important part of the most advanced artificial intelligence software we have today. By studying volumes of text, word embeddings are able to elicit meaning from the words within training data. Your goal is to train a word embedding on three famous works of Shakespeare to determine how well your embedding can understand the meaning of character names and other Shakespearean English words found in these plays.
## 1.1 Load Data
Use nltk.corpus.gutenberg.raw to load the three plays listed above into a single variable and lower the case.

In [1]:
import re
import numpy as np
import nltk
from difflib import SequenceMatcher
from nltk import sent_tokenize
from nltk.corpus import gutenberg, stopwords, wordnet as wn
from nltk.stem import WordNetLemmatizer
from autocorrect import Speller
from nltk import tokenize
from nltk.corpus import stopwords


for pkg in ["gutenberg", "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

In [2]:
play_ids = [
    "shakespeare-hamlet.txt",
    "shakespeare-macbeth.txt",
    "shakespeare-caesar.txt",
]

all_plays_text = " ".join(gutenberg.raw(play_id) for play_id in play_ids).lower()
print(f"Initial text length: {len(all_plays_text):,}")

Initial text length: 375,544


- Taking a look to the raw text:

In [3]:
print(all_plays_text)  # print the first 500 characters of the combined te

[the tragedie of hamlet by william shakespeare 1599]


actus primus. scoena prima.

enter barnardo and francisco two centinels.

  barnardo. who's there?
  fran. nay answer me: stand & vnfold
your selfe

   bar. long liue the king

   fran. barnardo?
  bar. he

   fran. you come most carefully vpon your houre

   bar. 'tis now strook twelue, get thee to bed francisco

   fran. for this releefe much thankes: 'tis bitter cold,
and i am sicke at heart

   barn. haue you had quiet guard?
  fran. not a mouse stirring

   barn. well, goodnight. if you do meet horatio and
marcellus, the riuals of my watch, bid them make hast.
enter horatio and marcellus.

  fran. i thinke i heare them. stand: who's there?
  hor. friends to this ground

   mar. and leige-men to the dane

   fran. giue you good night

   mar. o farwel honest soldier, who hath relieu'd you?
  fra. barnardo ha's my place: giue you goodnight.

exit fran.

  mar. holla barnardo

   bar. say, what is horatio there?
  hor. a peece of

## 1.2 Preprocessing steps:
- **Extra Step:** We decided to do text delimeters removal before tokenizing for better performance

In [4]:
def strip_boilerplate_and_noise(text: str) -> str:
    # Remove Gutenberg boilerplate if present
    text = re.sub(r"\*\*\*\s*start of.*?\*\*\*", " ", text, flags=re.I | re.S)
    text = re.sub(r"\*\*\*\s*end of.*?\*\*\*", " ", text, flags=re.I | re.S)

    # Remove bracketed stage directions, e.g., [enter ...]
    text = re.sub(r"\[[^\]]+\]", " ", text)

    # Remove act/scene headings
    text = re.sub(r"\b(act|scene)\s+[ivxlcdm]+\b", " ", text, flags=re.I)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_text = strip_boilerplate_and_noise(all_plays_text)
print(clean_text)

actus primus. scoena prima. enter barnardo and francisco two centinels. barnardo. who's there? fran. nay answer me: stand & vnfold your selfe bar. long liue the king fran. barnardo? bar. he fran. you come most carefully vpon your houre bar. 'tis now strook twelue, get thee to bed francisco fran. for this releefe much thankes: 'tis bitter cold, and i am sicke at heart barn. haue you had quiet guard? fran. not a mouse stirring barn. well, goodnight. if you do meet horatio and marcellus, the riuals of my watch, bid them make hast. enter horatio and marcellus. fran. i thinke i heare them. stand: who's there? hor. friends to this ground mar. and leige-men to the dane fran. giue you good night mar. o farwel honest soldier, who hath relieu'd you? fra. barnardo ha's my place: giue you goodnight. exit fran. mar. holla barnardo bar. say, what is horatio there? hor. a peece of him bar. welcome horatio, welcome good marcellus mar. what, ha's this thing appear'd againe to night bar. i haue seene no


- Tokenize the text into sentences, and then each sentence into words.

In [6]:
txt_sents = tokenize.sent_tokenize(clean_text)
txt_words = [tokenize.word_tokenize(sent) for sent in txt_sents]

- **Parameters Definition:** Here, we are defining the way different words sets (archaic words and stop words) are defined in order to preserve meaninful words when removing them in the next steps.

In [ ]:
# WordNet lemmas as a weak modern-English validity set
modern_vocab = {w.lower() for w in wn.all_lemma_names() if "_" not in w}

archaic_keep = {
    "thou", "thee", "thy", "thine", "doth", "hath", "art", "wilt", "shalt",
    "nay", "yea", "prythee", "hie", "wherefore", "hence", "whence", "oft",
    "methinks", "oer", "neer", "tis", "twas"
}

- Use Speller from the autocorrect library to correct spelling mistakes. 

In [8]:
spell = Speller(lang="en")
lemmatizer = WordNetLemmatizer()

def safe_spell(token: str) -> str:
    # Conservative strategy: only correct when we are fairly sure
    if token in archaic_keep or "'" in token or len(token) <= 3:
        return token
    if token in modern_vocab:
        return token

    cand = spell(token)
    if cand == token:
        return token

    # Accept only close edits into known modern vocab
    if cand in modern_vocab and SequenceMatcher(None, token, cand).ratio() >= 0.82:
        return cand
    return token



- Create a list of stopwords (using publicly available lists and/or adding your own) and remove these.

In [9]:
base_stop = set(stopwords.words("english"))
# Keep some function words that matter for Shakespeare semantics
stop_keep = {"not", "nor", "no", "thee", "thou", "thy", "thine", "ye", "you", "your"}
# Add corpus artifacts often not useful for semantics
stop_add = {"act", "scene", "enter", "exit", "exeunt"}
stop_set = (base_stop | stop_add) - stop_keep


- Use PorterStemmer or WordNetLemmatizer from nltk.stem on the text.
- Use regular expressions (the re library) to do any additional cleanup of the text you wish to do.

In [ ]:
def preprocess_sentence(sent: str):
    # Keep alphabetic tokens and internal apostrophes
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", sent.lower())

    # Required Speller step (conservative)
    tokens = [safe_spell(t) for t in tokens]

    # Required stopword step
    tokens = [t for t in tokens if t not in stop_set]

    # Required stemming/lemmatization step (choose one: here lemmatization)
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    # Additional cleanup
    tokens = [t for t in tokens if len(t) > 1 and not t.isdigit()]
    return tokens

processed_sentences = [preprocess_sentence(s) for s in txt_sents]
processed_sentences = [s for s in processed_sentences if s]  # drop empty

Processed sentences: [['act', 'primus'], ['scoena', 'prima'], ['barnardo', 'francisco', 'two', 'sentinel'], ['barnardo'], ["who's"], ['fran'], ['nay', 'answer', 'stand', 'unfold', 'your', 'self', 'bar'], ['long', 'liue', 'king', 'fran'], ['barnardo'], ['bar'], ['fran'], ['you', 'come', 'carefully', 'vpon', 'your', 'houre', 'bar'], ['ti', 'strook', 'twelve', 'get', 'thee', 'bed', 'francisco', 'fran'], ['releefe', 'much', 'thanks', 'ti', 'bitter', 'cold', 'sick', 'heart', 'barn'], ['haue', 'you', 'quiet', 'guard'], ['fran'], ['not', 'mouse', 'stirring', 'barn'], ['well', 'goodnight'], ['you', 'meet', 'ratio', 'marcellus', 'riuals', 'watch', 'bid', 'make', 'hast'], ['ratio', 'marcellus'], ['fran'], ['think'], ['stand', "who's"], ['hor'], ['friend', 'ground', 'mar'], ['leige', 'men', 'dane', 'fran'], ['giue', 'you', 'good', 'night', 'mar'], ['farwel', 'honest', 'soldier', 'hath', "relieu'd", 'you'], ['fra'], ['barnardo', "ha's", 'place', 'giue', 'you', 'goodnight'], ['fran'], ['mar'], ['ho

## 1.3 Processed Words
Print out the words in the first five sentences of the processed text data. (Viewing this may give you additional ideas for the previous steps.)

In [16]:
for i in range(5):
    print(f"Processed sentence # {i+1}: {processed_sentences[i]}")

Processed sentence # 1: ['act', 'primus']
Processed sentence # 2: ['scoena', 'prima']
Processed sentence # 3: ['barnardo', 'francisco', 'two', 'sentinel']
Processed sentence # 4: ['barnardo']
Processed sentence # 5: ["who's"]


# 2. Modeling
## 2.1 CBOW
Create a CBOW word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data. Use gensim.model.wv.key_to_index  and gensim.model.wv.get_vecattr to print out a list of the 20 most frequent words in the vocabulary along with the word count. Consider improving the text cleaning steps above based on this information. 

In [ ]:
from gensim.models import word2vec
np.random.seed(1)
model = word2vec.Word2Vec(processed_sentences,
                          vector_size=100,
                          epochs = 100,
                          window=5,
                          min_count=1,
                          workers=4)

In [ ]:
print('Printing CBOW Results')
for i, word in enumerate(model.wv.key_to_index):
    if i >= 20:
        break
    count = model.wv.get_vecattr(word, 'count')
    print(f"{word}: {count}")

you: 1115
not: 721
your: 528
haue: 444
ham: 337
thou: 307
lord: 306
shall: 300
no: 297
come: 283
king: 248
caesar: 230
good: 218
mac: 205
thy: 202
let: 196
make: 180
one: 178
thee: 174
know: 170


## 2.2 Skip Gram
Create a skipgram word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data.

In [24]:
model_sg = word2vec.Word2Vec(processed_sentences, 
                             vector_size=100,
                             window=5,
                             epochs=100,
                             min_count=1,
                             workers=4,
                             sg=1)

In [25]:
print('Printing Skip-Gram Results')
for i, word in enumerate(model_sg.wv.key_to_index):
    if i >= 20:
        break
    count = model_sg.wv.get_vecattr(word, 'count')
    print(f"{word}: {count}")


Printing Skip-Gram Results
you: 1115
not: 721
your: 528
haue: 444
ham: 337
thou: 307
lord: 306
shall: 300
no: 297
come: 283
king: 248
caesar: 230
good: 218
mac: 205
thy: 202
let: 196
make: 180
one: 178
thee: 174
know: 170


## 2.3 GloVe Model
Load the pretrained GloVe model from gensim.models.keyedvectors for comparison with the models trained on Shakespeare text. Use markdown to make note of the data that GloVe has been trained on.

In [28]:
from gensim.models.keyedvectors import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

glove_input_file = 'glove.6B.100d.txt'
word2vec_output_file = 'glove.6B.100d.w2vformat.txt'
glove2word2vec(glove_input_file, word2vec_output_file)

C:\Users\danie\AppData\Local\Temp\ipykernel_29884\3174052968.py:6: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input_file, word2vec_output_file)


(400000, 100)

In [29]:
glove_model = KeyedVectors.load_word2vec_format("glove.6B.100d.w2vformat.txt", binary=False)

### Data Used to Train GloVe
The GloVe (Global Vectors for Word Representation) model used in this comparison was trained on the **Wikipedia 2014 + Gigaword 5** corpus. This dataset contains:
 
- **6 billion tokens**  
- **400,000 vocabulary words**  
- A mixture of high‑quality, general‑purpose English text  
- Content spanning news articles, encyclopedic entries, and formal writing  
 
Because GloVe is trained on large, diverse, real‑world text, it provides a strong baseline for comparing word embeddings learned from the much smaller and stylistically unique Shakespeare corpus.

# 3. Discussion
## 3.1 Model Comparison
Compare the three models by finding the 5 most similar terms to each of the following terms: 'hamlet', 'cauldron', 'nature', 'spirit', 'general', and 'prythee'. Comment on how well each model captured the meaning of the word, and if there are multiple meanings, which meaning was given.

In [ ]:
terms = ['hamlet', 'cauldron', 'nature', 'spirit', 'general', 'prythee']
def printer(results):
    for i in results:
        print(i)
# Testing CBOW:
for term in terms:
    print(f"CBOW results for '{term}':")
    printer(model.wv.most_similar(term, topn=5))
    print()

CBOW results for 'hamlet':
('accord', 0.6507495045661926)
("vnforc'd", 0.6398200392723083)
('sowing', 0.607477068901062)
('bespeake', 0.6055812239646912)
('lunacie', 0.589656412601471)
CBOW results for 'cauldron':
('fenny', 0.9576291441917419)
('fillet', 0.946092426776886)
('bubble', 0.897608757019043)
('dragon', 0.8890260457992554)
('maw', 0.8652065396308899)
CBOW results for 'nature':
('eternity', 0.5441294312477112)
('insurrection', 0.5273669362068176)
('imaginings', 0.5243063569068909)
('bounteous', 0.5193246603012085)
('deprive', 0.518732488155365)
CBOW results for 'spirit':
('dumb', 0.48839786648750305)
('dismember', 0.48163577914237976)
('sable', 0.47730594873428345)
('diligence', 0.4636007249355316)
('cudgell', 0.4604714810848236)
CBOW results for 'general':
('cauiarie', 0.6355478763580322)
('coffer', 0.6032348871231079)
('concerne', 0.5722903609275818)
('ransom', 0.568087637424469)
('dollar', 0.5589064359664917)
CBOW results for 'prythee':
("prai'st", 0.6861351132392883)
('con

In [ ]:
# Testing Skip Gram:
for term in terms:
    print(f"Skip-Gram results for '{term}':")
    printer(model_sg.wv.most_similar(term, topn=5))
    print()

Skip-Gram results for 'hamlet':
('gen', 0.6808350682258606)
('tugging', 0.6679588556289673)
('beckens', 0.6605321764945984)
('sowing', 0.651596188545227)
('lunacie', 0.6442462205886841)
Skip-Gram results for 'cauldron':
('fenny', 0.8857555389404297)
('fillet', 0.8801769614219666)
('gaines', 0.8776977062225342)
('toile', 0.8405559062957764)
('dragon', 0.8130286931991577)
Skip-Gram results for 'nature':
("coppie's", 0.6217955350875854)
('boundlesse', 0.5939859747886658)
('eternity', 0.5782870054244995)
('perturbation', 0.5745630264282227)
('intemperance', 0.5725192427635193)
Skip-Gram results for 'spirit':
('dismember', 0.6206634640693665)
('conclude', 0.6130003333091736)
("scorn'd", 0.6039261817932129)
('ruffle', 0.5949006080627441)
("vntyr'd", 0.5896568298339844)
Skip-Gram results for 'general':
('cauiarie', 0.6438335180282593)
('sighe', 0.630115807056427)
('concerne', 0.6252549886703491)
('coffer', 0.6165291666984558)
('grudge', 0.6035159230232239)
Skip-Gram results for 'prythee':
("p

In [36]:
# Testing GloVe:
for term in terms:
    if term not in glove_model:
        print(f"'{term}' not found in GloVe model.")
        continue
    else:
        print(f"GloVe results for '{term}':")
        printer(glove_model.most_similar(term, topn=5))
        print()

GloVe results for 'hamlet':
('village', 0.6998987197875977)
('town', 0.6558532118797302)
('situated', 0.5926076769828796)
('located', 0.5660547614097595)
('unincorporated', 0.5599358677864075)

GloVe results for 'cauldron':
('caldron', 0.7603139281272888)
('flame', 0.6907342672348022)
('lit', 0.5912409424781799)
('torch', 0.5581893920898438)
('candle', 0.547653079032898)

GloVe results for 'nature':
('natural', 0.7198401093482971)
('true', 0.714996337890625)
('aspects', 0.7124009132385254)
('life', 0.7034883499145508)
('view', 0.6960832476615906)

GloVe results for 'spirit':
('passion', 0.744317889213562)
('faith', 0.7212756872177124)
('love', 0.686446487903595)
('sense', 0.672412633895874)
('devotion', 0.6691749691963196)

GloVe results for 'general':
('secretary', 0.7606937885284424)
('chief', 0.7243471145629883)
('gen.', 0.6899148225784302)
('president', 0.6798182129859924)
('vice', 0.6726594567298889)

'prythee' not found in GloVe model.


## 3.2 Cosine Similarity
Compare the three models by finding the cosine similarity between the following pairs of terms: ('brutus', 'murder'), ('lady macbeth', 'queen gertrude'), ('fortinbras', 'norway'), ('rome', 'norway'), ('ghost', 'spirit'), ('macbeth', 'hamlet'). Comment on how well each model captured the similarity between these terms, especially considering the data that each was trained on.

In [38]:
pairs = [
    ("brutus", "murder"),
    ("lady macbeth", "queen gertrude"),
    ("fortinbras", "norway"),
    ("rome", "norway"),
    ("ghost", "spirit"),
    ("macbeth", "hamlet"),
]

def phrase_vec(kv, phrase):
    toks = phrase.lower().split()
    toks = [t for t in toks if t in kv]
    if not toks:
        return None
    return np.mean([kv[t] for t in toks], axis=0)

def cos(kv, a, b):
    va = phrase_vec(kv, a)
    vb = phrase_vec(kv, b)
    if va is None or vb is None:
        return np.nan
    return float(kv.cosine_similarities(va, np.array([vb]))[0])

In [40]:
for a, b in pairs:
        print("pair: "f"({a}, {b})")
        print("CBOW: " + str(cos(model.wv, a, b)))
        print("SkipGram: " + str(cos(model_sg.wv, a, b)))
        print("GloVe: " + str(cos(glove_model, a, b)))
        print()

pair: (brutus, murder)
CBOW: 0.03669195622205734
SkipGram: 0.16054870188236237
GloVe: 0.07364358007907867

pair: (lady macbeth, queen gertrude)
CBOW: 0.23751193284988403
SkipGram: 0.3157860040664673
GloVe: 0.6617528200149536

pair: (fortinbras, norway)
CBOW: 0.5942118167877197
SkipGram: 0.5302016139030457
GloVe: -0.02896195836365223

pair: (rome, norway)
CBOW: 0.002962576225399971
SkipGram: 0.05091290548443794
GloVe: 0.28583669662475586

pair: (ghost, spirit)
CBOW: 0.0816466435790062
SkipGram: 0.24167896807193756
GloVe: 0.4282088875770569

pair: (macbeth, hamlet)
CBOW: 0.14663560688495636
SkipGram: 0.22223913669586182
GloVe: 0.429358571767807



## 3.3 Comparision of Results obtained from Linear Combinations
Compare the three models by finding the 5 most similar terms to each of the following word vectors obtained via linear combination: 'denmark' + 'queen', 'scotland' + 'army' + 'general', 'father' - 'man' + 'woman', 'mother' - 'woman' + 'man'. Comment on how well each model described the ideas behind these word vectors.

In [44]:
# 1) denmark + queen
print("pair: (denmark, queen)")
print("CBOW:", model.wv.most_similar(positive=['denmark', 'queen'], topn=5))
print("SkipGram:", model_sg.wv.most_similar(positive=['denmark', 'queen'], topn=5))
print("GloVe:", glove_model.most_similar(positive=['denmark', 'queen'], topn=5))
print()
# 2) scotland + army + general
print("pair: (scotland, army, general)")
print("CBOW:", model.wv.most_similar(positive=['scotland', 'army', 'general'], topn=5))
print("SkipGram:", model_sg.wv.most_similar(positive=['scotland', 'army', 'general'], topn=5))
print("GloVe:", glove_model.most_similar(positive=['scotland', 'army', 'general'], topn=5))
print()
# 3) father - man + woman
print("pair: (father - man +woman)")
print("CBOW:", model.wv.most_similar(positive=['father', 'woman'], negative=['man'], topn=5))
print("SkipGram:", model_sg.wv.most_similar(positive=['father', 'woman'], negative=['man'], topn=5))
print("GloVe:", glove_model.most_similar(positive=['father', 'woman'], negative=['man'], topn=5))
print()

# 4) mother - woman + man
print('pair: (mother - woman + man)')
print("CBOW:", model.wv.most_similar(positive=['mother', 'man'], negative=['woman'], topn=5))
print("SkipGram:", model_sg.wv.most_similar(positive=['mother', 'man'], negative=['woman'], topn=5))
print("GloVe:", glove_model.most_similar(positive=['mother', 'man'], negative=['woman'], topn=5))
print()

pair: (denmark, queen)
CBOW: [('polonius', 0.7521303296089172), ('vnckle', 0.6837482452392578), ('claudius', 0.658927857875824), ('successive', 0.6370890736579895), ('sainted', 0.6340468525886536)]
SkipGram: [('claudius', 0.7272992730140686), ('embracing', 0.7186902165412903), ('louingly', 0.7161023616790771), ('secricie', 0.6679055690765381), ('sainted', 0.6518247723579407)]
GloVe: [('sweden', 0.7461869120597839), ('norway', 0.7017143964767456), ('kingdom', 0.6878639459609985), ('princess', 0.6799803376197815), ('britain', 0.6786327362060547)]

pair: (scotland, army, general)
CBOW: [('casing', 0.6841856241226196), ("cabin'd", 0.6716322302818298), ('carbuncle', 0.6495980620384216), ('ransom', 0.6456170082092285), ('creep', 0.6431864500045776)]
SkipGram: [('coffer', 0.6525775790214539), ('sighe', 0.6466795802116394), ('ransom', 0.6306917667388916), ('foysons', 0.6238006353378296), ('doubted', 0.6027024388313293)]
GloVe: [('force', 0.7447022199630737), ('british', 0.7336236834526062), ('

## 3.4 Overall Comment on Performance
